In [23]:
import numpy as np
import pandas as pd

class Normalizer():
    def normalize(self,method,column):
        if method=='min_max':
            min_col=column.min()
            max_col=column.max()
            return (column-min_col)/(max_col-min_col)
        elif method=='z_score':
            mean_col=column.mean()
            sd_col=column.std()
            return (column-mean_col)/sd_col
        elif method=='decimal_scaling':
            factor=10**np.ceil(np.log10(np.abs(column).max()+1e-10))
            return column/factor
        else:
            raise ValueError('Invalid normalization method')

class DataFrame(Normalizer):
    def normalize(self,data,method='min_max',columns=None):
        if not isinstance(data,pd.DataFrame):
            raise TypeError('Data must be a Pandas DataFrame')
        if columns is None:
            columns=data.select_dtypes(include=[np.number]).columns

        result=data.copy()
        for col in columns:
            result[col]=super().normalize(method,data[col].fillna(data[col].mean()))
        return result

class Array(Normalizer):
    def normalize(self,data,method='min_max',columns=None):
        if not isinstance(data,np.ndarray):
            raise TypeError('Data must be a Numpy array')
        data=np.atleast_2d(data)

        normalized_data=data.copy()
        if columns is None:
            columns=range(normalized_data.shape[1])
        for col in columns:
            normalized_data[:,col]=super().normalize(method,normalized_data[:,col])
        return normalized_data
        
if __name__=='__main__':
    df_normalizer=DataFrame()
    df=pd.DataFrame({
        'A':[20,40,60],
        'B':[80,np.nan,120],
        'C':['J','K','L']})
    
    print('Original Pandas DataFrame:')
    print(df)
    print('\nNormalized Pandas DataFrame using Min-Max Method:')
    print(df_normalizer.normalize(df,method='min_max'))
    print('\nNormalized Pandas DataFrame using Z-Score Method:')
    print(df_normalizer.normalize(df,method='z_score'))
    print('\nNormalized Pandas DataFrame using Decimal Scaling Method:')
    print(df_normalizer.normalize(df,method='decimal_scaling'))
    
    np_normalizer=Array()
    arr=np.array([[20,40,60],
                 [80,100,120]])

    print('\nOriginal Numpy Array:')
    print(arr)    
    print('\nNormalized Numpy Array using Min-Max Method:')
    print(np_normalizer.normalize(arr,method='min_max'))
    print('\nNormalized Numpy Array using Z-Score Method:')
    print(np_normalizer.normalize(arr,method='z_score'))
    print('\nNormalized Numpy Array using Decimal Scaling Method:')
    print(np_normalizer.normalize(arr,method='decimal_scaling'))
    

Original Pandas DataFrame:
    A      B  C
0  20   80.0  J
1  40    NaN  K
2  60  120.0  L

Normalized Pandas DataFrame using Min-Max Method:
     A    B  C
0  0.0  0.0  J
1  0.5  0.5  K
2  1.0  1.0  L

Normalized Pandas DataFrame using Z-Score Method:
     A    B  C
0 -1.0 -1.0  J
1  0.0  0.0  K
2  1.0  1.0  L

Normalized Pandas DataFrame using Decimal Scaling Method:
     A     B  C
0  0.2  0.08  J
1  0.4  0.10  K
2  0.6  0.12  L

Original Numpy Array:
[[ 20  40  60]
 [ 80 100 120]]

Normalized Numpy Array using Min-Max Method:
[[0 0 0]
 [1 1 1]]

Normalized Numpy Array using Z-Score Method:
[[-1 -1 -1]
 [ 1  1  1]]

Normalized Numpy Array using Decimal Scaling Method:
[[0 0 0]
 [0 0 0]]
